In [4]:
!pip install -U bitsandbytes>=0.46.1

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset


/home/prateek/Prateek/LaunchPad/week7/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

MODEL_NAME   = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
TRAIN_PATH   = "/home/prateek/Prateek/LaunchPad/week8/Day2/data/train.jsonl"
VAL_PATH     = "/home/prateek/Prateek/LaunchPad/week8/Day2/data/validate.jsonl"
OUTPUT_DIR   = "/home/prateek/Prateek/LaunchPad/week8/Day2/adapters"

In [3]:
# =================>>> # LOAD MODEL IN 4-BIT (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


Loading weights: 100%|██████████| 201/201 [00:43<00:00,  4.63it/s, Materializing param=model.norm.weight]                              


In [4]:
## ========>> PREPARE MODEL FOR KBIT TRAINING
model = prepare_model_for_kbit_training(model)
# This does two things:
# - Freezes all 4-bit quantised layers
# - Casts layer norms to float32 for stable training

# ─────────────────────────────────────────
# 4. ATTACH LoRA ADAPTERS
# ─────────────────────────────────────────
lora_config = LoraConfig(
    r=16,                    # rank — higher = more capacity, more memory
    lora_alpha=32,           # scaling = alpha/r = 2x (standard)
    target_modules=[         # which layers to apply LoRA to
        "q_proj", "k_proj",
        "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected output: trainable params: ~1% of total


trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


In [5]:
dataset = load_dataset(
    "json",
    data_files={"train": TRAIN_PATH, "validation": VAL_PATH}
)

Generating train split: 855 examples [00:00, 86485.71 examples/s]
Generating validation split: 95 examples [00:00, 81434.47 examples/s]


In [ ]:
# ─────────────────────────────────────────
# 6. TRAINING ARGUMENTS
# ─────────────────────────────────────────
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=0.05,
    fp16=True,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none",
    max_length=512,
    dataset_text_field="text",
    packing=False,
)

In [7]:
# ─────────────────────────────────────────
# 7. TRAINER
# ─────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
)

Truncating eval dataset: 100%|██████████| 95/95 [00:00<00:00, 27582.64 examples/s]


In [8]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.
/home/prateek/Prateek/LaunchPad/week7/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss


KeyboardInterrupt: 